# Feature Scaling or Normalization

_Feature Engineering_

**Put features that live on wildly different scales onto a common footing — so distance- and gradient-based models stop being bullied by the big-numbered column.**

A feature's units are arbitrary. Whether you record a length in millimeters or kilometers does not change the underlying information, yet it changes the raw number by a factor of a million. Many models cannot tell the difference between "this feature carries more signal" and "this feature just has bigger numbers".

       Scaling removes the units. It rewrites every feature into a comparable range so that the model judges features by their information, not their magnitude. Crucially, all three methods here are affine per-feature (min-max, standardization) or per-row rescaling (L2) — they move and stretch points but never bend the cloud. The shape of each feature's histogram is preserved; only its location and spread change.

---

This notebook is a practice scaffold for the **Feature Scaling or Normalization** lesson. We will take it one step at a time: see the scaling problem on real data, then walk the three classic recipes line by line. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_wine

data = load_wine(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## See it for yourself: the problem, then the fix

Run this cell. It plots the **raw** data and reproduces the problem it causes, then plots the **engineered** data and shows the fix — with before/after numbers.

### Step 1 — Load two features on wildly different scales

k-Nearest Neighbors classifies a point by its closest neighbors, where "closest" means smallest Euclidean distance — and distance sums the gaps over *every* feature. If one feature has big numbers and another has small ones, the big-number feature alone decides the distance.

We load the wine dataset and pick two real features that show this off: `proline` runs into the hundreds, while `hue` sits around 1. We use a fixed train/test split so the numbers reproduce every run.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

wine = load_wine()
feat_names = list(wine.feature_names)
i_proline = feat_names.index("proline")   # big scale
i_hue = feat_names.index("hue")           # small scale
X = wine.data[:, [i_proline, i_hue]]      # columns: [proline, hue]
y = wine.target

# Fixed split so the numbers reproduce every run.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=0, stratify=y
)

# Show just how different the scales are (this is the whole point).
print(f"proline range: {X[:,0].min():7.1f} .. {X[:,0].max():7.1f}  (hundreds)")
print(f"hue     range: {X[:,1].min():7.2f} .. {X[:,1].max():7.2f}  (about 1)")

### Step 2 — Reproduce the problem on raw features

Fit a plain k-NN (k=5) on the raw columns. Because proline's numbers dwarf hue's, the distance is essentially just the difference in proline — hue is ignored even though it carries real signal. The accuracy comes out weak.

We also build the standardized version now (subtract each feature's mean, divide by its std) so we can plot raw vs scaled side by side in the next step.

In [ ]:
# Same k-NN, raw features. Distance is basically just |proline difference|,
# because proline numbers dwarf hue numbers. hue is ignored -> weak accuracy.
knn_raw = KNeighborsClassifier(n_neighbors=5)
knn_raw.fit(X_train, y_train)
acc_raw = accuracy_score(y_test, knn_raw.predict(X_test))
print(f"\nPROBLEM  k-NN on RAW features: accuracy = {acc_raw:.3f}")

# Standardize: for each feature subtract its mean, divide by its std.
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)
X_s = scaler.transform(X)

### Step 3 — Visualize raw vs standardized

Plot the two clouds side by side. On the raw axes the cloud is stretched along proline — distances are dominated by the horizontal axis. After standardizing, both features span roughly the same range, so the cloud looks round and both axes can contribute to the distance. We force equal aspect on the right so the roundness is honest.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))

# Left — RAW data: the cloud is stretched along proline.
for c in np.unique(y):
    ax[0].scatter(X[y == c, 0], X[y == c, 1], s=18, alpha=0.7,
                  label=wine.target_names[c])
ax[0].set_title("RAW: cloud STRETCHED along proline")
ax[0].set_xlabel("proline (hundreds)")
ax[0].set_ylabel("hue (~1)")
ax[0].legend(fontsize=8)

# Right — STANDARDIZED data: round cloud, axes comparable.
for c in np.unique(y):
    ax[1].scatter(X_s[y == c, 0], X_s[y == c, 1], s=18, alpha=0.7,
                  label=wine.target_names[c])
ax[1].set_title("STANDARDIZED: round cloud, axes comparable")
ax[1].set_xlabel("proline (standardized)")
ax[1].set_ylabel("hue (standardized)")
ax[1].set_aspect("equal", "box")
ax[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

### Step 4 — Show the fix

Run the *same* k-NN (k=5), now on the standardized features. Both features get a fair vote in the distance, so the accuracy jumps. The one-line summary makes the before/after contrast explicit.

In [ ]:
# SAME k-NN (k=5), now on standardized features. Both features get a fair vote
# in the distance -> accuracy jumps.
knn_fix = KNeighborsClassifier(n_neighbors=5)
knn_fix.fit(X_train_s, y_train)
acc_fix = accuracy_score(y_test, knn_fix.predict(X_test_s))
print(f"FIX      k-NN on STANDARDIZED features: accuracy = {acc_fix:.3f}")

# One-line before/after summary.
print(f"\nPROBLEM (raw): {acc_raw:.3f}   →   FIX (standardized): {acc_fix:.3f}")

## Reference implementation — scikit-learn

### Step 1 — Load the data and apply min-max scaling

This is the book's worked example on the Online News Popularity dataset (download it from the book's repo, linked in the comment). We scale `n_tokens_content`, the number of words in an article.

**Min-max scaling** maps every value linearly into the range [0, 1]: the smallest value becomes 0, the largest becomes 1. `minmax_scale` expects a 2-D array, so we pass a single-column DataFrame.

In [ ]:
import pandas as pd
from sklearn import preprocessing

# Online News Popularity dataset (UCI). Get it from the book's repo:
#   github.com/alicezheng/feature-engineering-book
df = pd.read_csv('OnlineNewsPopularity.csv')
df.columns = [c.strip() for c in df.columns]   # the CSV has leading spaces

# 1. MIN-MAX SCALING -> every value mapped into [0, 1].
df['minmax'] = preprocessing.minmax_scale(df[['n_tokens_content']])

### Step 2 — Standardize, then L2-normalize

Two more recipes on the same column.

**Standardization** (z-score / variance scaling) subtracts the mean and divides by the standard deviation, giving mean 0 and variance 1.

**L2 normalization** divides by an L2 norm. Here `axis=0` normalizes down the column, so on this single feature each value just gets divided by the column's norm. On a full feature matrix you would instead normalize each *row* so every example's vector has length 1 (sits on the unit sphere).

In [ ]:
# 2. STANDARDIZATION (variance scaling / z-score) -> mean 0, variance 1.
df['standardized'] = (
    preprocessing.StandardScaler().fit_transform(df[['n_tokens_content']])
)

# 3. L2 NORMALIZATION -> divide each ROW vector by its L2 norm (unit sphere).
#    (Per-row: here on a single column it just maps each value to +/-1, but on a
#     full feature matrix it makes every example's vector have length 1.)
df['l2_normalized'] = preprocessing.normalize(
    df[['n_tokens_content']], norm='l2', axis=0
)

print(df[['n_tokens_content', 'minmax', 'standardized', 'l2_normalized']].head())

### Step 3 — Confirm scaling changes the range, not the shape

Plot the original column and two of its scaled versions as histograms. The key takeaway: the bars have the *same shape* in all three panels — only the x-axis units differ. Scaling is an affine map; it shifts and stretches the axis but never bends the distribution.

In [ ]:
# Scaling changes the RANGE, not the SHAPE: plot to confirm the histograms
# have identical shape, only different x-axis units.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(6, 8))

df['n_tokens_content'].hist(ax=axes[0], bins=100)
axes[0].set_title('Original n_tokens_content')

df['minmax'].hist(ax=axes[1], bins=100)
axes[1].set_title('Min-max scaled to [0, 1]')

df['standardized'].hist(ax=axes[2], bins=100)
axes[2].set_title('Standardized (mean 0, var 1)')

plt.tight_layout()
plt.show()

## Visualize the data & results

_Two real features on wildly different scales: wine 'proline' (~hundreds) vs 'hue' (~1). What do their min / mean / max look like BEFORE standardization, and how does standardization put them on a common footing?_

### Step 1 — Read the two features' raw min / mean / max

Pull `proline` and `hue` from the wine dataset again and report each one's minimum, mean, and maximum *before* any scaling. The numbers make the scale gap concrete: proline ranges over hundreds while hue hovers near 1, and their means are orders of magnitude apart.

In [ ]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler

d = load_wine()
names = list(d.feature_names)
pi = names.index('proline')
hi = names.index('hue')

proline = d.data[:, pi]   # amino acid, ~hundreds
hue = d.data[:, hi]       # color ratio, ~1

def mmm(a):  # min, mean, max
    return round(a.min(), 3), round(a.mean(), 3), round(a.max(), 3)

print('BEFORE proline min/mean/max:', mmm(proline))  # (278.0, 746.893, 1680.0)
print('BEFORE hue     min/mean/max:', mmm(hue))       # (0.48, 0.957, 1.71)

### Step 2 — Standardize and re-read the same statistics

Standardize both features together, then print the same min/mean/max. After standardization every feature has mean 0 (the printed value rounds to -0.0 / 0.0) and a comparable spread, so the two columns now sit on a common footing — exactly what distance- and gradient-based models want.

In [ ]:
# Standardize each feature: subtract mean, divide by std -> mean 0, variance 1.
Z = StandardScaler().fit_transform(np.c_[proline, hue])

print('AFTER  proline min/mean/max:', mmm(Z[:, 0]))   # (-1.493, -0.0, 2.971)
print('AFTER  hue     min/mean/max:', mmm(Z[:, 1]))   # (-2.095, 0.0, 3.302)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You feed a k-nearest-neighbours classifier two features: annual income (tens of thousands) and number of children (0-5). Without scaling, the model is essentially ignoring the number of children. Explain why, and fix it.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write the Euclidean distance between two people across the two raw features. — _Distance squared is (income difference)^2 + (children difference)^2; income differences are in the thousands, children differences at most 5._
- Observe that the income term dwarfs the children term by a factor of millions once squared. — _k-NN finds 'nearest' points almost entirely by income, so number of children barely affects who counts as a neighbor._
- Standardize both features (subtract mean, divide by standard deviation), fit on the training set only. — _Now both features have variance 1, so they contribute comparably to the distance and the model can use both._

**Answer:** k-NN uses Euclidean distance, and income's raw magnitude (tens of thousands) overwhelms children (0-5), so 'nearest' is decided almost entirely by income. Standardize each feature to mean 0, variance 1 (fit the scaler on training data only, inside a Pipeline). Then both features contribute comparably and the model uses number of children too.

</details>

**Problem 2.** A teammate runs StandardScaler().fit_transform(X) on the full dataset, THEN splits into train and test and reports a cross-validation AUC. Why is the reported score optimistic, and what is the correct protocol?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify that the scaler's mean and standard deviation were computed using the test rows too. — _Information about the held-out rows (their statistics) has flowed into the transform applied to the training rows._
- Name this as data leakage: the validation rows are no longer truly unseen. — _The model gets a peek at the held-out distribution, so the validation score overstates real-world performance._
- Move the scaler inside a Pipeline and fit it within each cross-validation fold, on the training fold only. — _A Pipeline re-fits the scaler on each fold's training data, so the held-out fold's statistics never leak in._

**Answer:** Fitting the scaler on all of X lets the test rows' mean and standard deviation leak into the training transform — that is leakage, and it inflates the AUC. Correct protocol: put StandardScaler inside a Pipeline so it is fit on the training fold only within each cross-validation split. The honest score will usually be a bit lower.

</details>

**Problem 3.** You have a right-skewed feature (a long tail of large values). A colleague says "just standardize it and the skew will go away." Is that right? What actually removes the skew, and in what order relative to scaling?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that standardization is an affine map: subtract a constant, divide by a constant. — _Affine maps only shift and stretch the axis; they cannot bend the distribution or shorten a tail._
- Conclude the histogram's shape (the skew) is unchanged by standardization — only its center and spread move. — _A right-skewed feature is still right-skewed by the same amount after standardizing._
- Apply a log or power transform FIRST to compress the tail, THEN standardize the result for the model. — _The non-linear log/power transform reshapes the distribution; scaling afterward puts it on a comparable footing._

**Answer:** No — standardization is an affine (shift-and-scale) map, so it leaves the distribution's shape, including the skew, unchanged. To remove skew you need a non-linear log or power transform, applied BEFORE scaling. Then standardize the transformed feature for the model.

</details>

**Problem 4.** You are about to put a StandardScaler in front of a gradient-boosted tree model. A reviewer says it is pointless. Are they right? What kind of model WOULD need the scaler here?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall how a tree splits: it asks 'is feature x below a threshold t?'. — _The split depends only on the ordering of values, not their absolute magnitude._
- Note that any increasing rescaling maps the threshold along with the values, so the same rows fall on each side. — _Monotone rescaling leaves every split — and therefore the whole tree — identical, so scaling does nothing._
- Identify the models that DO need scaling: linear/SGD/neural-net and any distance-based model (k-NN, k-means, RBF-SVM). — _Those use gradients or distances, which are sensitive to feature magnitude, unlike threshold splits._

**Answer:** The reviewer is right: tree-based models split on thresholds, and any monotone rescaling moves the threshold with the values, leaving the splits unchanged — so scaling is wasted work for gradient-boosted trees. Scaling matters for gradient-based models (linear with regularization, SGD, neural nets) and distance-based models (k-NN, k-means, RBF-SVM).

</details>